# Hand Gesture Recognition: Rock, Paper, Scissors

**Project:** PRODIGY_ML_04  
**Goal:** Classify an image of a hand gesture as `paper`, `rock`, or `scissors` using a convolutional neural network (CNN).

This notebook keeps the original image-folder approach and completes the missing stages: data inspection, preprocessing, model building, training, evaluation, visualizations, sample predictions, and model saving.

> **Dataset requirement:** The dataset folder must contain one subfolder per class, for example:
>
> ```text
> rps-cv-images/
> ├── paper/
> ├── rock/
> └── scissors/
> ```
>
> Each subfolder should contain images belonging to that class.

## 1. Import libraries

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from pathlib import Path
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

## 2. Set project parameters and dataset path

**Important:** The original notebook used an absolute Windows path. Keep that path if it is correct on your laptop; otherwise change `dataset_path` to the location of your `rps-cv-images` folder.

In [ ]:
dataset_path = Path(r"C:\Users\divya\OneDrive\PRODIGY\PRODIGY_ML_04\rps-cv-images")

# If your dataset is in the same folder as this notebook, you can instead use:
# dataset_path = Path("rps-cv-images")

img_size = (150, 150)
batch_size = 32
seed = 42
epochs = 15

print("Dataset path:", dataset_path)
print("Dataset folder exists:", dataset_path.exists())

if not dataset_path.exists():
    raise FileNotFoundError(
        "Dataset folder not found. Update dataset_path in this cell to the folder "
        "that contains the paper, rock and scissors subfolders."
    )

## 3. Check the dataset folders

In [ ]:
class_folders = sorted(
    [folder.name for folder in dataset_path.iterdir() if folder.is_dir()]
)

print("Detected classes:", class_folders)
print("Number of classes:", len(class_folders))

if len(class_folders) < 2:
    raise ValueError(
        "Expected class subfolders (for example paper, rock, scissors). "
        "Check that dataset_path points to the dataset root folder."
    )

for folder in class_folders:
    image_count = sum(
        1 for file in (dataset_path / folder).rglob("*")
        if file.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
    )
    print(f"{folder}: {image_count} images")

## 4. Load training and validation datasets

The dataset is split into 80% training and 20% validation data. Both calls use the same seed and validation split so they select matching subsets.

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    str(dataset_path),
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="int"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    str(dataset_path),
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="int"
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("Class names:", class_names)
print("Number of classes:", num_classes)

if class_names != val_ds.class_names:
    raise ValueError("Training and validation class names do not match.")

## 5. Display sample images

In [ ]:
plt.figure(figsize=(10, 8))

for images, labels in train_ds.take(1):
    n = min(9, len(images))
    for i in range(n):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")

plt.tight_layout()
plt.show()

## 6. Optimize the input pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.shuffle(1000, seed=seed).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

## 7. Build the CNN model

The model uses data augmentation to help reduce overfitting, followed by convolution and pooling layers that learn visual patterns in the hand images.

In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomZoom(0.10),
    ],
    name="data_augmentation"
)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(img_size[0], img_size[1], 3)),
    data_augmentation,
    tf.keras.layers.Rescaling(1.0 / 255),

    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

model.summary()

## 8. Compile the model

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 9. Train the model

Early stopping restores the weights from the best validation-loss epoch if validation performance stops improving.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=callbacks
)

## 10. Plot training and validation accuracy

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], marker="o", label="Training accuracy")
plt.plot(history.history["val_accuracy"], marker="o", label="Validation accuracy")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

## 11. Plot training and validation loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], marker="o", label="Training loss")
plt.plot(history.history["val_loss"], marker="o", label="Validation loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

## 12. Evaluate the model

In [ ]:
val_loss, val_accuracy = model.evaluate(val_ds, verbose=1)

print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy * 100:.2f}%")

## 13. Generate predictions and classification report

In [ ]:
y_true = []
y_pred = []

for images, labels in val_ds:
    probabilities = model.predict(images, verbose=0)
    predictions = np.argmax(probabilities, axis=1)

    y_true.extend(labels.numpy().tolist())
    y_pred.extend(predictions.tolist())

print(classification_report(
    y_true,
    y_pred,
    labels=list(range(num_classes)),
    target_names=class_names,
    zero_division=0
))

## 14. Confusion matrix

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(num_classes))
)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(7, 6))
display.plot(cmap="Blues", ax=ax, values_format="d")
plt.title("Confusion Matrix - Hand Gesture Recognition")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 15. View example predictions

In [ ]:
plt.figure(figsize=(12, 8))
shown = 0

for images, labels in val_ds.take(1):
    probabilities = model.predict(images, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    confidences = np.max(probabilities, axis=1)

    n = min(9, len(images))
    for i in range(n):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))

        actual_name = class_names[int(labels[i])]
        predicted_name = class_names[int(predictions[i])]
        confidence = confidences[i] * 100

        plt.title(
            f"Actual: {actual_name}\nPredicted: {predicted_name} ({confidence:.1f}%)",
            fontsize=9
        )
        plt.axis("off")
        shown += 1

plt.tight_layout()
plt.show()

## 16. Predict a new image

Change `image_path` to the path of a hand image on your computer. The image should show one of the gesture classes used in training.

In [ ]:
image_path = "path_to_your_hand_image.jpg"  # Replace with your image path

if Path(image_path).exists():
    image = tf.keras.utils.load_img(image_path, target_size=img_size)
    image_array = tf.keras.utils.img_to_array(image)
    image_array = tf.expand_dims(image_array, axis=0)

    probabilities = model.predict(image_array, verbose=0)[0]
    predicted_index = int(np.argmax(probabilities))

    print("Predicted gesture:", class_names[predicted_index])
    print(f"Confidence: {probabilities[predicted_index] * 100:.2f}%")

    plt.imshow(image)
    plt.title(f"Prediction: {class_names[predicted_index]}")
    plt.axis("off")
    plt.show()
else:
    print("Set image_path to a valid image file to test a new image.")

## 17. Save the trained model and class names

In [ ]:
model.save("hand_gesture_recognition.keras")

with open("class_names.json", "w") as file:
    json.dump(class_names, file)

print("Model and class names saved successfully.")

## 18. Conclusion

This project builds a convolutional neural network to classify hand gesture images into the classes available in the dataset (such as paper, rock, and scissors). The workflow includes image loading, preprocessing, data augmentation, model training, validation, and performance evaluation using accuracy, loss, a classification report, and a confusion matrix.

**Limitations:** The model classifies still images from the same classes represented in its training dataset. It is not a real-time webcam gesture recognizer. Performance on new backgrounds, lighting conditions, camera angles, or people may differ from validation performance. A separate held-out test dataset would provide a stronger final evaluation.